# RECAST on a synthetic timeline

Follow one question from its answer options to selected frames. Scripted text responses and cached probabilities make each decision inspectable on CPU. The live checkpoint adapters are in `src/backends.py`.

In [1]:
from pathlib import Path
import json, math, sys
import torch
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/reformulation.py').exists())
sys.path.insert(0, str(root))
from src.model import recast, rosin_threshold, select_frames
from src.reformulation import reformulate

## Query reformulation · §III-B

> Synthesize always produces the final description.

The fixture requests both optional stages. Its first draft fails format checks, so the second synthesis receives the violations.

In [2]:
question = 'When the music stops, which item does the child pick up?'
options = ['a red ball', 'a blue kite', 'a green bag', 'a toy train']
description = 'A child in stripes holding either a red ball, a blue kite, a green bag, or a toy train.'
calls = []
def scripted(stage, instruction, payload):
    calls.append(stage)
    if stage == 'route':
        return json.dumps(dict(need_ground=True, need_decompose=True))
    if stage == 'ground':
        return 'Use the visible pickup action as evidence.'
    if stage == 'decompose':
        return ', '.join(options)
    if 'violations' not in payload:
        return '{}'
    return json.dumps(dict(description=description, option_spans=options))

result = reformulate(question, options, scripted)
print(result)

A child in stripes holding either a red ball, a blue kite, a green bag, or a toy train.


In [3]:
assert result == description
assert calls == ['route', 'ground', 'decompose', 'synthesize', 'synthesize']

## Rosin threshold · §III-C1

> All frames with $s_i \geq \tau$ are retained as candidates.

With 27 low scores and three peaks, the histogram corner falls between the low mass and the three plausible frames.

In [4]:
scores = torch.full((30,), 0.01, dtype=torch.float64)
scores[[2, 3, 24]] = torch.tensor([0.97, 0.96, 0.85], dtype=scores.dtype)
threshold = rosin_threshold(scores)
pool = (scores >= threshold).nonzero().flatten().tolist()
print(float(threshold), pool)

0.025 [2, 3, 24]


In [5]:
assert 0.01 < threshold < 0.85
assert pool == [2, 3, 24]

## Temporal MMR · §III-C2, Eq. 1

> the max aggregation captures the collision with the nearest selected frame.

Recompute the redundancy against every selected frame as an independent check on the incremental implementation. At $\lambda=1$ only relevance contributes.

In [6]:
def reference(pool, scores, k, weight, sigma):
    chosen = [max(pool, key=lambda i: float(scores[i]))]
    while len(chosen) < k:
        def objective(i):
            penalty = max(math.exp(-((i-j)/sigma)**2 / 2) for j in chosen)
            return weight * float(scores[i]) - (1-weight) * penalty
        chosen.append(max((i for i in pool if i not in chosen), key=objective))
    return sorted(chosen)

for weight in (0, 0.3, 0.6, 1):
    for k in (1, 2, 3):
        actual = select_frames(scores, k, weight, 5).tolist()
        assert actual == reference(pool, scores, k, weight, 5)
print(select_frames(scores, 2, 0.3, 5).tolist())

[2, 24]


In [7]:
assert select_frames(scores, 2, 1, 5).tolist() == [2, 3]
assert select_frames(scores, 2, 0.3, 5).tolist() == [2, 24]
assert len(set(select_frames(scores, 7, 0.3, 5).tolist())) == 7
assert select_frames([0.5]*4, 2, 0.3, 5).tolist() == [0, 3]
assert select_frames([0.1], 32, 0.3, 5).tolist() == [0]

## One query per frame · §III-B4

> the encoder still receives exactly one description

Count all frame–description pairs at the scoring boundary. The resulting offsets index the original 1-FPS timeline.

In [8]:
pairs = []
def cached_score(frames, text):
    pairs.extend((frame, text) for frame in frames)
    return scores
frames = list(range(len(scores)))
text, curve, offsets = recast(question, options, frames, scripted, cached_score,
                             k=2, relevance=0.3, sigma=5)
print(text, offsets.tolist(), sep='\n')

A child in stripes holding either a red ball, a blue kite, a green bag, or a toy train.
[2, 24]


In [9]:
assert pairs == [(i, description) for i in frames]
assert torch.equal(curve, scores)
assert offsets.tolist() == [2, 24]

## Interpreting the result

Compacting the three surviving frame indices to `[0, 1, 2]` would erase the 21-second gap between the second and third candidates. Preserve the original indices. The assertions check control flow and numerical selection. Assessing visual relevance and QA accuracy requires running the pretrained models on benchmark videos.